# pptk-revived Examples

Examples demonstrating pptk-revived features for point cloud visualisation.

> **Note:** The viewer opens in a separate window. In headless environments set `DISPLAY` or use `Xvfb`.

In [1]:
import numpy as np
import pptk

_viewers = []  # track all open windows for cleanup at the end

## 1. Basic viewer

In [2]:
xyz = pptk.rand(100, 3)
v = pptk.viewer(xyz)
v.set(point_size=0.005)
_viewers.append(v)

## 2. Shading by a scalar attribute

Pass a 1-D array as second argument to shade points by a scalar value.

In [3]:
v = pptk.viewer(xyz, xyz[:, 2])  # shade by height (Z)
v.set(point_size=0.005)
_viewers.append(v)

## 3. RGB coloring

Pass an (N, 3) float32 array with values in [0, 1] for per-point RGB colour.

In [4]:
rgb = pptk.rand(100, 3)
v = pptk.viewer(xyz, rgb)
v.set(point_size=0.005)
_viewers.append(v)

## 4. Multiple attributes

Load several attribute sets at once. Press `[` / `]` in the viewer to cycle between them.

In [5]:
scalar = xyz[:, 2]           # height
rgb    = pptk.rand(100, 3)   # random colours

v = pptk.viewer(xyz)
v.attributes(scalar, rgb)    # load both; cycle with [ / ]
v.set(point_size=0.005)
_viewers.append(v)

## 5. k-Nearest Neighbour queries

Use `pptk.points` to build a k-d tree and query k-NN neighbourhoods,
then colour by local point density.

In [6]:
pts = pptk.rand(5000, 3)           # pptk.Points wraps a k-d tree
nbhds = pts.nbhds(k=10)            # list of index arrays, one per point

pts_np = np.asarray(pts)
mean_dist = np.array([
    np.linalg.norm(pts_np[idx] - pts_np[i], axis=1).mean()
    for i, idx in enumerate(nbhds)
], dtype=np.float32)

v = pptk.viewer(pts_np, mean_dist)
v.set(point_size=0.008)
v.color_map('hot')
_viewers.append(v)

## 6. Normal estimation

Estimate per-point normals using PCA on local neighbourhoods, then colour by the Z component.

> Always use `r=float('inf')` for pure k-NN (omitting the radius cutoff avoids a divide-by-zero
> when a point has no neighbours within the given radius).

In [ ]:
rng = np.random.default_rng(0)
n = 5000
xy = rng.random((n, 2))
z  = np.sin(xy[:, 0] * 6 * np.pi) * np.cos(xy[:, 1] * 6 * np.pi) * 0.05
surface = np.column_stack([xy, z])

# r=float('inf') → pure k-NN, no radius cutoff; num_procs=1 → single-threaded
normals = pptk.estimate_normals(surface, k=15, r=float('inf'), num_procs=1)
nz = normals[:, 2].astype(np.float32)

v = pptk.viewer(surface, nz)
v.set(point_size=0.006)
v.color_map('cool')
_viewers.append(v)

## 7. Parametric surface — Möbius strip

In [ ]:
u = np.linspace(0, 2 * np.pi, 400)
v_vals = np.linspace(-0.5, 0.5, 80)
U, V = np.meshgrid(u, v_vals)
U, V = U.ravel(), V.ravel()

x = (1 + V * np.cos(U / 2)) * np.cos(U)
y = (1 + V * np.cos(U / 2)) * np.sin(U)
z = V * np.sin(U / 2)

mobius = np.column_stack([x, y, z]).astype(np.float32)
hue    = (U / (2 * np.pi)).astype(np.float32)

v = pptk.viewer(mobius, hue)
v.set(point_size=0.005)
v.color_map('hsv')
_viewers.append(v)

## 8. Camera animation

Use `v.play()` to animate the camera along a path of poses `(x, y, z, phi, theta, r)`.

In [ ]:
rng2 = np.random.default_rng(1)
cloud = rng2.random((5000, 3)).astype(np.float32)

v = pptk.viewer(cloud)
v.set(point_size=0.01)

n_frames = 8
poses = [
    (0.5, 0.5, 0.5, 2 * np.pi * i / n_frames, np.deg2rad(20), 1.5)
    for i in range(n_frames + 1)
]
v.play(poses, ts=np.linspace(0, 6, len(poses)), repeat=False, interp='cubic_natural')
_viewers.append(v)

## 9. Point selection

Hold **Ctrl + LMB drag** to select points in the viewer.  
Read back the selected indices with `v.get('selected')`.

In [ ]:
xyz2 = pptk.rand(1000, 3)
v = pptk.viewer(xyz2)
v.set(point_size=0.01)
_viewers.append(v)

# After selecting points interactively in the viewer window:
# selected = v.get('selected')
# print('Selected indices:', selected)

## Close all viewers

In [ ]:
for v in _viewers:
    v.close()
_viewers.clear()